In [4]:
import osmnx as ox
import geopandas as gpd
import networkx as nx
import pandas as pd
from shapely.geometry import Point, MultiPoint
import os, time
from tqdm.notebook import tqdm

BASE = "/home/jovyan/work/children15mc"
PROC = f"{BASE}/data/processed"
ISO_DIR = f"{PROC}/iso_by_borough"
os.makedirs(ISO_DIR, exist_ok=True)

ox.settings.use_cache = True
ox.settings.cache_folder = f"{BASE}/cache"

CRS = 27700
WALK_SPEED_KMH = 3.5
TRIP_MIN = 15
m_per_min = WALK_SPEED_KMH * 1000 / 60

def make_isochrone(G, x27700, y27700, trip_min=15, buff=30):
    pt = gpd.GeoSeries([Point(x27700, y27700)], crs=27700).to_crs(4326).iloc[0]
    node = ox.distance.nearest_nodes(G, pt.x, pt.y)
    sub_g = nx.ego_graph(G, node, radius=trip_min, distance="walk_time")
    pts = [Point(d["x"], d["y"]) for _, d in sub_g.nodes(data=True)]
    if len(pts) < 3:
        return None
    poly = gpd.GeoSeries(MultiPoint(pts), crs=4326).to_crs(27700)
    return poly.convex_hull.buffer(buff).iloc[0]

print("Configuration complete.")

Configuration complete.


In [8]:
df = gpd.read_file(f"{PROC}/analysis_table.gpkg")
boroughs = sorted(df["lad22nm"].unique())
print(f"Total {len(boroughs)} boroughs, {len(df)} LSOAs")

for bname in boroughs:
    safe = bname.replace(" ", "_").replace(",", "")
    outfile = f"{ISO_DIR}/{safe}.gpkg"

    # Skip completed items
    if os.path.exists(outfile):
        print(f"{bname} Completed, skipped")
        continue

    t0 = time.time()
    sub = df[df["lad22nm"] == bname].copy()

    # Download road network
    poly_27700 = sub.geometry.union_all().buffer(500)
    poly_4326 = gpd.GeoSeries([poly_27700], crs=27700).to_crs(4326).iloc[0]
    try:
        G = ox.graph_from_polygon(poly_4326, network_type="walk", simplify=True)
    except Exception as e:
        print(f"{bname} Failed to download road network: {e}")
        continue

    for u, v, k, d in G.edges(keys=True, data=True):
        d["walk_time"] = d["length"] / m_per_min

    # Run for all LSOAs in the borough
    rows = []
    for _, row in sub.iterrows():
        geom = make_isochrone(G, row.cent_x, row.cent_y, TRIP_MIN)
        rows.append({"lsoa21cd": row.lsoa21cd, "geometry": geom})

    iso = gpd.GeoDataFrame(rows, crs=CRS).dropna(subset=["geometry"])
    iso.to_file(outfile, driver="GPKG")
    print(f"✓ {bname}: {len(iso)}/{len(sub)}, took {time.time()-t0:.0f}s")

print("All done!")

Total 33 boroughs, 4659 LSOAs
Barking and Dagenham Completed, skipped
Barnet Completed, skipped
Bexley Completed, skipped
Brent Completed, skipped
Bromley Completed, skipped
Camden Completed, skipped
City of London Completed, skipped
Croydon Completed, skipped
Ealing Completed, skipped
Enfield Completed, skipped
Greenwich Completed, skipped
Hackney Completed, skipped
Hammersmith and Fulham Completed, skipped
Haringey Completed, skipped
Harrow Completed, skipped
Havering Completed, skipped
Hillingdon Completed, skipped
Hounslow Completed, skipped
Islington Completed, skipped
Kensington and Chelsea Completed, skipped
Kingston upon Thames Completed, skipped
Lambeth Completed, skipped
Lewisham Completed, skipped
Merton Completed, skipped
Newham Completed, skipped
Redbridge Completed, skipped
Richmond upon Thames Completed, skipped
Southwark Completed, skipped
Sutton Completed, skipped
Tower Hamlets Completed, skipped
Waltham Forest Completed, skipped
Wandsworth Completed, skipped
Westminst

In [6]:
import geopandas as gpd, pandas as pd, glob, os

BASE = "/home/jovyan/work/children15mc"
ISO_DIR = f"{BASE}/data/processed/iso_by_borough"

files = glob.glob(f"{ISO_DIR}/*.gpkg")

# Merge the isochrones for all boroughs and count the total
all_iso = pd.concat([gpd.read_file(f) for f in files], ignore_index=True)
all_iso = gpd.GeoDataFrame(all_iso, crs=27700)
print(f"Total isochronous cycles: {len(all_iso)}")

all_iso.to_file(f"{BASE}/data/processed/isochrones_all.gpkg", driver="GPKG")

Total isochronous cycles: 4659
